# COCO digital channel — resumable

29 reviewed decoders. Use the completed production embedding bank; no new embedding.

Each method: save attacks in temporary ZIPs → decode saved PNGs → save results on Drive → remove attack ZIPs. Start with five images. No Base64 code.

Use **Linux + NVIDIA**, preferably the same A100 runtime as the reviewed bank; high RAM. Download the matching source ZIP too. No GPU runtime is guaranteed to last overnight.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, zipfile

N_IMAGES = 5                  # After review: change ONLY this to 100; keep the same output.
MODELS = "all"                # All 29; or a list of exact method IDs for diagnostics.
MAX_HOURS = 16                # A session budget, NOT a guarantee of Colab uptime.
SSD_RESERVE_GIB = 60          # Leave SSD room for one isolated decoder environment.
PAIRWISE_TABLES = True
SOURCE_ARCHIVE = Path("COCO_Digital_Channel_RESUMABLE_source.zip")

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if os.name != "posix":
    raise RuntimeError("Full 29-method execution requires Linux/NVIDIA: use Colab or a GPU server.")
if IN_COLAB:
    drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive") if IN_COLAB else Path("/workspace/data")
BANK_ROOT = BASE / "watermarking_production_bank_v1"       # READ ONLY: completed embedding bank.
OUTPUT_ROOT = BASE / "watermarking_digital_channel_v1"    # Durable measurements and reports.
SCRATCH_ROOT = Path("/content/wmbench_digital_scratch") if IN_COLAB else Path("/workspace/wmbench_digital_scratch")
SETTINGS = dict(n_images=N_IMAGES, models=MODELS, max_hours=MAX_HOURS, device="cuda",
    bank=str(BANK_ROOT), output=str(OUTPUT_ROOT), scratch=str(SCRATCH_ROOT),
    ssd_reserve_gib=SSD_RESERVE_GIB, pairwise=PAIRWISE_TABLES, download_author_assets=True,
    manual_assets_root=str(BANK_ROOT / "_manual_assets"))
print("Bank (read-only):", BANK_ROOT)
print("Results:", OUTPUT_ROOT)
print("Temporary fallback:", OUTPUT_ROOT / "_temporary")
print("Per method: save 78 attacked PNGs per available carrier in ZIPs → decode → commit → remove ZIPs.")


## 1. Install the orchestrator and verify its source


In [ ]:
# Upload the matching small source ZIP once; a copy is kept on Drive.
EXPECTED_SHA = "fbaec01ed86f1b0293250e9cde03aca8d1a9a63a0c0a2c69b761b0d1d74577ff"
SAVED_ZIP = OUTPUT_ROOT / "_release" / SOURCE_ARCHIVE.name
if not SOURCE_ARCHIVE.is_file() and SAVED_ZIP.is_file():
    SOURCE_ARCHIVE = SAVED_ZIP
if not SOURCE_ARCHIVE.is_file():
    if not IN_COLAB:
        raise FileNotFoundError("Place the matching source ZIP beside this notebook or set SOURCE_ARCHIVE.")
    from google.colab import files
    uploaded = files.upload()
    matches = [Path(name) for name in uploaded if hashlib.sha256(uploaded[name]).hexdigest() == EXPECTED_SHA]
    if len(matches) != 1:
        raise ValueError("Upload exactly the source ZIP supplied with this notebook.")
    SOURCE_ARCHIVE = matches[0]
data = SOURCE_ARCHIVE.read_bytes()
if hashlib.sha256(data).hexdigest() != EXPECTED_SHA:
    raise ValueError("Source ZIP checksum mismatch. Do not use a different notebook's ZIP.")
SAVED_ZIP.parent.mkdir(parents=True, exist_ok=True)
if not SAVED_ZIP.exists():
    SAVED_ZIP.write_bytes(data)
if hashlib.sha256(SAVED_ZIP.read_bytes()).hexdigest() != EXPECTED_SHA:
    raise ValueError("Saved release readback failed")
PROJECT = SCRATCH_ROOT / "release"
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
    for member in archive.infolist():
        target = (PROJECT / member.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()) or member.is_dir():
            raise ValueError("Unsafe or unexpected source ZIP entry")
        content = archive.read(member)
        if target.exists() and target.read_bytes() != content:
            raise ValueError(f"Source changed: {target}. Use a different empty SCRATCH_ROOT; retain OUTPUT_ROOT.")
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.15"], check=True)
import uv
UV = str(uv.find_uv_bin())
controller_dir = SCRATCH_ROOT / "controller"
CONTROLLER = controller_dir / "bin/python"
if not CONTROLLER.exists():
    subprocess.run([UV, "venv", "--python", "3.10.12", str(controller_dir)], check=True)
# This is the CPU attack/report + quality-metric environment, NOT a model environment.
subprocess.run([UV, "pip", "install", "--python", str(CONTROLLER),
    "numpy==1.26.4", "Pillow==10.4.0", "scipy==1.11.4", "scikit-image==0.22.0",
    "psutil==6.1.1", "gdown==5.2.0", "torch==2.5.1", "torchvision==0.20.1", "lpips==0.1.4"], check=True)
ENV = os.environ.copy()
ENV.update(PYTHONPATH=os.pathsep.join([str(PROJECT / "src"), str(PROJECT / "production/src")]),
    PYTHONUNBUFFERED="1", OMP_NUM_THREADS="4", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="4",
    PATH=str(Path(UV).parent)+os.pathsep+ENV.get("PATH", ""),
    UV_PYTHON_INSTALL_DIR=str(SCRATCH_ROOT / "python"))
CONFIG = SCRATCH_ROOT / "digital-settings.json"
CONFIG.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
print("Free SSD: %.1f GiB" % (shutil.disk_usage(SCRATCH_ROOT).free / 1024**3))
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)


## 2. Restore the reviewed system dependencies

Only missing MagickWand/GL libraries are installed; no method is changed.


In [ ]:
# Only RoSteALS/SepWater need these author-import system libraries.
# Restore the setup used by RoSteALS_AUTHOR_FIX and RoSteALS_SepWater_RECOVERY.
# No model source, checkpoint, Python lock, payload or matching rule is changed.
import ctypes
import ctypes.util

def ensure_author_system_dependencies(models, in_colab):
    affected = {"rosteals", "sepwater"}
    selected = affected if models == "all" else affected.intersection(models)
    if not selected:
        return {}

    requirements = {
        "libmagickwand-dev": (
            ("MagickWand-6.Q16", "MagickWand-6.Q16HDRI", "MagickWand-7.Q16HDRI",
             "MagickWand-7.Q16", "MagickWand"), "MagickWandGenesis"),
        "libgl1": (("GL",), None),
    }

    def inspect_libraries():
        found = {}
        for package, (names, symbol) in requirements.items():
            for name in names:
                path = ctypes.util.find_library(name)
                if not path:
                    continue
                try:
                    library = ctypes.CDLL(path)
                    if symbol:
                        getattr(library, symbol)
                except (OSError, AttributeError):
                    continue
                found[package] = path
                break
        return found

    found = inspect_libraries()
    missing = [package for package in requirements if package not in found]
    if missing:
        install = ["apt-get", "-qq", "install", "-y", "--no-install-recommends", *missing]
        if not in_colab or not shutil.which("apt-get") or os.geteuid() != 0:
            raise RuntimeError(
                "RoSteALS/SepWater system libraries are missing. On Debian/Ubuntu run: "
                "sudo apt-get update && sudo " + " ".join(install)
                + ". Then rerun this setup cell; keep the existing output directory.")
        print("Installing missing system libraries for " + ", ".join(sorted(selected))
              + ": " + ", ".join(missing), flush=True)
        subprocess.run(["apt-get", "-qq", "update"], check=True, timeout=600)
        subprocess.run(install, check=True, timeout=600)
        found = inspect_libraries()
        missing = [package for package in requirements if package not in found]
        if missing:
            raise RuntimeError("System-library preflight failed after installation: "
                               + ", ".join(missing) + ". No model has been started.")
    print("RoSteALS/SepWater system-library preflight passed:", found, flush=True)
    return found

AUTHOR_SYSTEM_LIBRARIES = ensure_author_system_dependencies(MODELS, IN_COLAB)


In [ ]:
subprocess.run([str(CONTROLLER), "-m", "wmbench_digital.runner", "validate", "--settings", str(CONFIG)], env=ENV, check=True)


## 3. Run

Keep the same output after disconnects. Valid results are never retried to improve BER. If one method fails technically, the run stops and retains its temporary ZIPs. Logs and completed rows are on Drive.


In [ ]:
# Stop any previous runner for this output folder before starting.
# After a disconnect rerun the same cells with the same bank/output and source ZIP.
command = [str(CONTROLLER), "-m", "wmbench_digital.runner", "run", "--settings", str(CONFIG)]
process = subprocess.Popen(command, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, encoding="utf-8", errors="replace", start_new_session=True)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError("Runner stopped. See persistent per-method logs; completed records remain safe.")
except BaseException:
    if process.poll() is None:
        import signal
        os.killpg(process.pid, signal.SIGTERM)
        process.wait()
    raise


## 4. Reports

Report rebuilding does not rerun attacks or load a decoder. Partial coverage remains visible.


In [ ]:
subprocess.run([str(CONTROLLER), "-m", "wmbench_digital.runner", "report", "--settings", str(CONFIG)],
    env=ENV, check=True)
from IPython.display import Markdown, display
display(Markdown((OUTPUT_ROOT / "_reports/REPORT.md").read_text(encoding="utf-8")))
print("Aggregate tables:", OUTPUT_ROOT / "_reports")
print("Each method's tables:", OUTPUT_ROOT / "<method>/tables")
print("Detailed bits, errors and attack parameters:", OUTPUT_ROOT / "<method>/records/*.json.gz")
print("Copy OUTPUT_ROOT to your SSD after completion. Temporary attack ZIPs are intentionally not permanent.")
